In [8]:
!pip install gitpython

In [15]:
from git import Repo

In [21]:
Repo_Url = "https://github.com/PhonePe/pulse.git"
Clone_Path = "Phonepe_clonedata"
Repo.clone_from (Repo_Url,Clone_Path)
print("Cloned data successful")

Cloned data successful


In [26]:
import os            # check folder structure
for root,dirs,files in os.walk("Phonepe_clonedata"):
    print(root)
    break

Phonepe_clonedata


In [61]:
import os # import required libraries
import json
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

In [63]:
conn = psycopg2.connect(   # Create database
    host="localhost",
    user="postgres",
    password="qwerty123",
    port="5432"
)

conn.autocommit=True
cursor=conn.cursor()

cursor.execute("CREATE DATABASE phonepe_db;")

print("✅ Database Created")

✅ Database Created


In [64]:
# Connect Project Database (ONLY ONCE)
conn = psycopg2.connect(
    host="localhost",
    user="postgres",
    password="qwerty123",
    database="phonepe_db",
    port="5432"
)

cursor = conn.cursor()

print("✅ Connected to phonepe_db")

✅ Connected to phonepe_db


In [65]:
# Create ALL 9 Tables
create_tables = """

CREATE TABLE IF NOT EXISTS aggregated_transaction(
state TEXT,year INT,quarter INT,
transaction_type TEXT,
transaction_count BIGINT,
transaction_amount FLOAT);

CREATE TABLE IF NOT EXISTS aggregated_user(
state TEXT,year INT,quarter INT,
user_type TEXT,
user_count BIGINT);

CREATE TABLE IF NOT EXISTS aggregated_insurance(
state TEXT,year INT,quarter INT,
insurance_type TEXT,
insurance_count BIGINT,
insurance_amount FLOAT);

CREATE TABLE IF NOT EXISTS map_transaction(
state TEXT,year INT,quarter INT,
district TEXT,
transaction_count BIGINT,
transaction_amount FLOAT);

CREATE TABLE IF NOT EXISTS map_user(
state TEXT,year INT,quarter INT,
district TEXT,
registered_users BIGINT,
app_opens BIGINT);

CREATE TABLE IF NOT EXISTS map_insurance(
state TEXT,year INT,quarter INT,
district TEXT,
insurance_count BIGINT,
insurance_amount FLOAT);

CREATE TABLE IF NOT EXISTS top_transaction(
state TEXT,year INT,quarter INT,
district TEXT,
transaction_count BIGINT,
transaction_amount FLOAT);

CREATE TABLE IF NOT EXISTS top_user(
state TEXT,year INT,quarter INT,
district TEXT,
registered_users BIGINT);

CREATE TABLE IF NOT EXISTS top_insurance(
state TEXT,year INT,quarter INT,
district TEXT,
insurance_count BIGINT,
insurance_amount FLOAT);
"""

cursor.execute(create_tables)
conn.commit()

print("✅ All Tables Created")

✅ All Tables Created


In [66]:
# COMMON FUNCTION (Bulk Loader)
def load_table(df, table, columns):
    query=f"INSERT INTO {table} ({columns}) VALUES %s"
    execute_values(cursor, query, df.values.tolist())
    conn.commit()
    print(f"✅ {table} loaded")

In [90]:
# load AGGREGATED TRANSACTION
path=r"Phonepe_clonedata\data\aggregated\transaction\country\india\state/"
data=[]

for s in os.listdir(path):
    for y in os.listdir(path+s+"/"):
        yp=path+s+"/"+y+"/"
        for f in os.listdir(yp):
            if f.endswith(".json"):
                j=json.load(open(yp+f))
                try:
                    for t in j["data"]["transactionData"]:
                        data.append([s,int(y),int(f[:-5]),
                        t["name"],
                        t["paymentInstruments"][0]["count"],
                        t["paymentInstruments"][0]["amount"]])
                except: pass

df=pd.DataFrame(data,columns=["state","year","quarter","transaction_type","transaction_count","transaction_amount"])

load_table(df,"aggregated_transaction",
"state,year,quarter,transaction_type,transaction_count,transaction_amount")

✅ aggregated_transaction loaded


In [92]:
# load AGGREGATED USER
path=r"Phonepe_clonedata\data\aggregated\user\country\india\state/"
data=[]

for s in os.listdir(path):
    for y in os.listdir(path+s+"/"):
        yp=path+s+"/"+y+"/"
        for f in os.listdir(yp):
            if f.endswith(".json"):
                j=json.load(open(yp+f))
                try:
                    for u in j["data"]["usersByDevice"]:
                        data.append([s,int(y),int(f[:-5]),
                        u["brand"],u["count"]])
                except: pass

df=pd.DataFrame(data,columns=["state","year","quarter","user_type","user_count"])

load_table(df,"aggregated_user",
"state,year,quarter,user_type,user_count")

✅ aggregated_user loaded


In [96]:
# AGGREGATED INSURANCE Load
path=r"Phonepe_clonedata\data\aggregated\insurance\country\india\state/"
data=[]

for s in os.listdir(path):
    for y in os.listdir(path+s+"/"):
        yp=path+s+"/"+y+"/"
        for f in os.listdir(yp):
            if f.endswith(".json"):
                j=json.load(open(yp+f))
                try:
                    for t in j["data"]["transactionData"]:
                        data.append([s,int(y),int(f[:-5]),
                        t["name"],
                        t["paymentInstruments"][0]["count"],
                        t["paymentInstruments"][0]["amount"]])
                except: pass

df=pd.DataFrame(data)

load_table(df,"aggregated_insurance",
"state,year,quarter,insurance_type,insurance_count,insurance_amount")

✅ aggregated_insurance loaded


In [97]:
# MAP TRANSACTION Load
path=r"Phonepe_clonedata\data\map\transaction\hover\country\india\state/"
data=[]

for s in os.listdir(path):
    for y in os.listdir(path+s+"/"):
        yp=path+s+"/"+y+"/"
        for f in os.listdir(yp):
            if f.endswith(".json"):
                j=json.load(open(yp+f))
                try:
                    for d in j["data"]["hoverDataList"]:
                        data.append([s,int(y),int(f[:-5]),
                        d["name"],
                        d["metric"][0]["count"],
                        d["metric"][0]["amount"]])
                except: pass

df=pd.DataFrame(data)

load_table(df,"map_transaction",
"state,year,quarter,district,transaction_count,transaction_amount")

✅ map_transaction loaded


In [98]:
# MAP USER load
path=r"Phonepe_clonedata\data\map\user\hover\country\india\state/"
data=[]

for s in os.listdir(path):
    for y in os.listdir(path+s+"/"):
        yp=path+s+"/"+y+"/"
        for f in os.listdir(yp):
            if f.endswith(".json"):
                j=json.load(open(yp+f))
                try:
                    for d,v in j["data"]["hoverData"].items():
                        data.append([s,int(y),int(f[:-5]),
                        d,v["registeredUsers"],v["appOpens"]])
                except: pass

df=pd.DataFrame(data)

load_table(df,"map_user",
"state,year,quarter,district,registered_users,app_opens")

✅ map_user loaded


In [99]:
# MAP INSURANCE Load
path=r"Phonepe_clonedata\data\map\insurance\hover\country\india\state/"
data=[]

for s in os.listdir(path):
    for y in os.listdir(path+s+"/"):
        yp=path+s+"/"+y+"/"
        for f in os.listdir(yp):
            if f.endswith(".json"):
                j=json.load(open(yp+f))
                try:
                    for d in j["data"]["hoverDataList"]:
                        data.append([s,int(y),int(f[:-5]),
                        d["name"],
                        d["metric"][0]["count"],
                        d["metric"][0]["amount"]])
                except: pass

df=pd.DataFrame(data)

load_table(df,"map_insurance",
"state,year,quarter,district,insurance_count,insurance_amount")

✅ map_insurance loaded


In [100]:
# TOP TRANSACTION Load
path=r"Phonepe_clonedata\data\top\transaction\country\india\state/"
data=[]

for s in os.listdir(path):
    for y in os.listdir(path+s+"/"):
        yp=path+s+"/"+y+"/"
        for f in os.listdir(yp):
            if f.endswith(".json"):
                j=json.load(open(yp+f))
                try:
                    for d in j["data"]["districts"]:
                        data.append([s,int(y),int(f[:-5]),
                        d["entityName"],
                        d["metric"]["count"],
                        d["metric"]["amount"]])
                except: pass

df=pd.DataFrame(data)

load_table(df,"top_transaction",
"state,year,quarter,district,transaction_count,transaction_amount")

✅ top_transaction loaded


In [101]:
# TOP USER Load
path=r"Phonepe_clonedata\data\top\user\country\india\state/"
data=[]

for s in os.listdir(path):
    for y in os.listdir(path+s+"/"):
        yp=path+s+"/"+y+"/"
        for f in os.listdir(yp):
            if f.endswith(".json"):
                j=json.load(open(yp+f))
                try:
                    for d in j["data"]["districts"]:
                        data.append([s,int(y),int(f[:-5]),
                        d["name"],d["registeredUsers"]])
                except: pass

df=pd.DataFrame(data)

load_table(df,"top_user",
"state,year,quarter,district,registered_users")

✅ top_user loaded


In [102]:
# TOP INSURANCE Load
path=r"Phonepe_clonedata\data\top\insurance\country\india\state/"
data=[]

for s in os.listdir(path):
    for y in os.listdir(path+s+"/"):
        yp=path+s+"/"+y+"/"
        for f in os.listdir(yp):
            if f.endswith(".json"):
                j=json.load(open(yp+f))
                try:
                    for d in j["data"]["districts"]:
                        data.append([s,int(y),int(f[:-5]),
                        d["entityName"],
                        d["metric"]["count"],
                        d["metric"]["amount"]])
                except: pass

df=pd.DataFrame(data)

load_table(df,"top_insurance",
"state,year,quarter,district,insurance_count,insurance_amount")

✅ top_insurance loaded


In [104]:
# Final Check
cursor.execute("""
SELECT table_name FROM information_schema.tables
WHERE table_schema='public';
""")

print(cursor.fetchall())

[('aggregated_transaction',), ('aggregated_user',), ('aggregated_insurance',), ('map_transaction',), ('map_user',), ('map_insurance',), ('top_transaction',), ('top_user',), ('top_insurance',)]


In [105]:
# Check Rows
cursor.execute("SELECT COUNT(*) FROM aggregated_transaction;")
print(cursor.fetchone())

(5034,)


In [106]:
print(df.shape)
df.head()

(5608, 6)


,0,1,2,3,4,5
0,andaman-&-nicobar-islands,2020,2,nicobars,3,565.0
1,andaman-&-nicobar-islands,2020,2,south andaman,3,795.0
2,andaman-&-nicobar-islands,2020,3,south andaman,35,13651.0
3,andaman-&-nicobar-islands,2020,3,nicobars,5,1448.0
4,andaman-&-nicobar-islands,2020,3,north and middle andaman,1,281.0


In [80]:
import os # check
os.listdir("Phonepe_clonedata")

['.git', '.gitignore', 'data', 'LICENSE', 'README.md']

In [81]:
os.listdir("Phonepe_clonedata/data") # check

['aggregated', 'map', 'top']

In [89]:
os.listdir(r"Phonepe_clonedata\data\aggregated\transaction\country\india\state/") # check

['andaman-&-nicobar-islands',
 'andhra-pradesh',
 'arunachal-pradesh',
 'assam',
 'bihar',
 'chandigarh',
 'chhattisgarh',
 'dadra-&-nagar-haveli-&-daman-&-diu',
 'delhi',
 'goa',
 'gujarat',
 'haryana',
 'himachal-pradesh',
 'jammu-&-kashmir',
 'jharkhand',
 'karnataka',
 'kerala',
 'ladakh',
 'lakshadweep',
 'madhya-pradesh',
 'maharashtra',
 'manipur',
 'meghalaya',
 'mizoram',
 'nagaland',
 'odisha',
 'puducherry',
 'punjab',
 'rajasthan',
 'sikkim',
 'tamil-nadu',
 'telangana',
 'tripura',
 'uttar-pradesh',
 'uttarakhand',
 'west-bengal']